# 🛰️ DynaLand & Deep Learning Spatio-Temporal Anomaly Detection
## Comparative Reproduction: Script Reference vs. Paper-Text Configurations
**Paper Reference:** *Sustainability 2023, 15(6), 4725* (https://doi.org/10.3390/su15064725)

This notebook reproduces the experimental workflows and validates both dataset configurations:
1. **Script Default Suite (DynaLand Repository):**
   - **Altamira:** MODIS MOD13Q1 (250m, NDVI)
   - **Brumadinho:** Sentinel-2 MSI (10m, NDWI & NDVI)
   - **Mariana:** Landsat-8 OLI (30m, GVMI & NDWI)
2. **Literal Paper Text Suite (Sustainability 15-04725 Section 4.2.1):**
   - **Altamira:** MODIS MOD13Q1 (250m, NDVI)
   - **Brumadinho:** Landsat-8 OLI (30m, NDVI)
   - **Mariana:** Sentinel-2 MSI (10m, NDWI)


In [ ]:
import os
import sys

# Detect if executing inside Google Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    REPO_URL = "https://github.com/Arkadyg27/TimeSeriesProject.git"
    PROJECT_DIR = "/content/TimeSeriesProject"

    os.chdir('/content')
    if not os.path.exists(PROJECT_DIR):
        !git clone {REPO_URL}
    else:
        os.chdir(PROJECT_DIR)
        !git pull

    os.chdir(PROJECT_DIR)
    print("Google Colab detected. Working directory set to:", os.getcwd())
else:
    print("Running locally. Working directory set to:", os.getcwd())


In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    !pip install mlflow rasterio pymannkendall tabulate
else:
    print("Local environment detected. Make sure dependencies are installed.")


In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules

# 1. Native Colab Authentication
if IN_COLAB:
    try:
        from google.colab import auth
        auth.authenticate_user()
    except Exception as e:
        print('Colab auth notice:', e)

import ee

# Team Earth Engine Projects
PROJECT_IDS = ['889258893131', 'timeseriesproject-503021']

initialized = False
for proj in PROJECT_IDS:
    try:
        ee.Initialize(project=proj)
        print(f'Earth Engine initialized successfully using project: "{proj}"')
        initialized = True
        break
    except Exception:
        continue

if not initialized:
    try:
        ee.Initialize()
        print('Earth Engine initialized using account default project!')
    except Exception:
        print('Prompting interactive Earth Engine authentication...')
        ee.Authenticate()
        ee.Initialize()


In [ ]:
import os, sys
IN_COLAB = 'google.colab' in sys.modules

# Sync full and partial download checkpoints from Google Drive to preserve download progress
if IN_COLAB:
    from google.colab import drive
    try:
        drive.mount('/content/drive', force_remount=False)
    except ValueError as e:
        if 'Mountpoint must not already contain files' in str(e):
            print('Detected dirty mountpoint. Cleaning up...')
            import shutil
            shutil.rmtree('/content/drive', ignore_errors=True)
            drive.mount('/content/drive', force_remount=True)
        else:
            raise
    drive_cache_dir = '/content/drive/MyDrive/Study/MSc_CE_BGU/Time Series Analysis/Final Project/TimeSeriesProject'

    if os.path.exists(drive_cache_dir):
        import glob, shutil
        for f in glob.glob(f"{drive_cache_dir}/*.parquet"):
            dest = f"/content/TimeSeriesProject/{os.path.basename(f)}"
            if not os.path.exists(dest) or os.path.getsize(dest) < 100000:
                shutil.copy2(f, dest)
        print('Smart-Synced dataset caches from Google Drive (preventing corrupt 0-byte files)!')
    else:
        !mkdir -p "/content/drive/MyDrive/Study/MSc_CE_BGU/Time Series Analysis/Final Project/TimeSeriesProject" 


In [ ]:
%env MLFLOW_ALLOW_FILE_STORE=true
import os, sys
import mlflow
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    try:
        drive.mount('/content/drive', force_remount=False)
    except ValueError as e:
        if 'Mountpoint must not already contain files' in str(e):
            print('Detected dirty mountpoint. Cleaning up...')
            import shutil
            shutil.rmtree('/content/drive', ignore_errors=True)
            drive.mount('/content/drive', force_remount=True)
        else:
            raise

    # Define project path in Google Drive
    DRIVE_PROJECT_PATH = '/content/drive/MyDrive/Study/MSc_CE_BGU/Time Series Analysis/Final Project/TimeSeriesProject'
    MLRUNS_DIR = os.path.join(DRIVE_PROJECT_PATH, 'mlruns')
    os.makedirs(MLRUNS_DIR, exist_ok=True)

    # Direct MLflow to log persistently to Google Drive
    os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
    if "MLFLOW_TRACKING_URI" in os.environ:
        del os.environ["MLFLOW_TRACKING_URI"]
    mlflow.set_tracking_uri(f"file:///{MLRUNS_DIR}")

    import IPython
    IPython.get_ipython().run_line_magic('env', f'MLFLOW_TRACKING_URI=file:///{MLRUNS_DIR}')
    print(f"MLflow tracking initialized! Runs will be saved to: {MLRUNS_DIR}")
else:
    print("Local environment: MLflow tracking locally.")


In [ ]:
# ========================================================
# SMART PREPROCESSING: Ingest and Center All Sensor Regions
# ========================================================
import os, shutil, sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    DRIVE_PREPROCESS = '/content/drive/MyDrive/Study/MSc_CE_BGU/Time Series Analysis/Final Project/TimeSeriesProject/Preprocess'
    LOCAL_PREPROCESS = 'Preprocess'

    # Check if preprocessed files already exist in Google Drive
    check_file = os.path.join(DRIVE_PREPROCESS, 'Altamira_NDVI_CenteredMatrix.parquet')

    if os.path.exists(check_file):
        print("Found preprocessed data in Google Drive! Linking to local workspace...")
        os.makedirs(LOCAL_PREPROCESS, exist_ok=True)
        for file_name in os.listdir(DRIVE_PREPROCESS):
            src = os.path.join(DRIVE_PREPROCESS, file_name)
            dst = os.path.join(LOCAL_PREPROCESS, file_name)
            if not os.path.exists(dst):
                try:
                    os.symlink(src, dst)
                except Exception:
                    shutil.copy2(src, dst)
        DRIVE_TIFF = os.path.join(DRIVE_PROJECT_PATH, "Tiff")
        LOCAL_TIFF = "Tiff"
        if os.path.exists(DRIVE_TIFF):
            if os.path.exists(LOCAL_TIFF) and not os.path.islink(LOCAL_TIFF):
                shutil.rmtree(LOCAL_TIFF)
            if not os.path.exists(LOCAL_TIFF):
                try:
                    os.symlink(DRIVE_TIFF, LOCAL_TIFF)
                except Exception:
                    shutil.copytree(DRIVE_TIFF, LOCAL_TIFF, dirs_exist_ok=True)
        print("All preprocessed data linked! Ready to run experiments.")
    else:
        print("First-time run: Preprocessing all dataset configurations...")
        !python run_preprocessing.py --suite all

        # Save a backup to Google Drive
        shutil.copytree(LOCAL_PREPROCESS, DRIVE_PREPROCESS, dirs_exist_ok=True)
        print("Preprocessed files backed up to Google Drive for future sessions!")
else:
    print("Local environment: Running preprocessing for all suites...")
    !python run_preprocessing.py --suite all


## 📊 Part 1: Script Default Reproduction (DynaLand Repository Code)
In this section, we run the baselines, classical ML sweeps, and Deep LSTM Autoencoder using the exact dataset configurations present in the author's open-source repository:
- **Altamira:** MODIS (NDVI)
- **Brumadinho:** Sentinel-2 (NDWI & NDVI)
- **Mariana:** Landsat-8 (GVMI & NDWI)


In [ ]:
# Run Baseline Z-Score (Leaky vs. Walk-Forward Leak-Free)
!python run_baseline_all.py --suite script

# Run Classical Machine Learning Sweeps (Isolation Forest & One-Class SVM)
!python Altamira_Modis_repro.py
!python Brumadinho_Sentinel_repro.py
!python Mariana_Landsat_repro.py


In [ ]:
# Train Batched GPU Accelerated LSTM Autoencoder on Script Datasets
!python train_deep.py --suite script --batch_size 512

# Compute and Log the Complete Unsupervised Spatial-Temporal Metrics Suite
!python compute_custom_metrics.py --suite script


## 📑 Part 2: Exact "Paper Text" Reproduction (Sustainability 15-04725)
In this section, we run the full experimental pipeline on the literal pairings stated in Section 4.2.1 of the paper text:
- **Altamira:** MODIS (NDVI)
- **Brumadinho:** Landsat-8 OLI (NDVI, 30m)
- **Mariana:** Sentinel-2 MSI (NDWI, 10m)


In [ ]:
# Run Baseline Z-Score on Paper-Text Datasets
!python run_baseline_all.py --suite paper_text

# Train Batched GPU Accelerated LSTM Autoencoder on Paper-Text Datasets
!python train_deep.py --suite paper_text --batch_size 512

# Compute and Log the Complete Unsupervised Metrics Suite on Paper-Text Datasets
!python compute_custom_metrics.py --suite paper_text


In [ ]:
import pandas as pd

print("=" * 80)
print("             PART 1: SCRIPT DEFAULT SUITE METRICS (DynaLand Code)           ")
print("=" * 80)
if os.path.exists("THREE_DATASETS_ALL_METRICS.csv"):
    df_script = pd.read_csv("THREE_DATASETS_ALL_METRICS.csv")
    display(df_script)
else:
    print("Run compute_custom_metrics.py --suite script to generate Part 1 metrics.")

print("
" + "=" * 80)
print("             PART 2: PAPER TEXT SUITE METRICS (Sustainability 15-04725)      ")
print("=" * 80)
if os.path.exists("PAPER_TEXT_ALL_METRICS.csv"):
    df_paper = pd.read_csv("PAPER_TEXT_ALL_METRICS.csv")
    display(df_paper)
else:
    print("Run compute_custom_metrics.py --suite paper_text to generate Part 2 metrics.")


In [ ]:
import sys, os
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    DRIVE_BACKUP = "/content/drive/MyDrive/Study/MSc_CE_BGU/Time Series Analysis/Final Project/TimeSeriesProject"
    !mkdir -p "{DRIVE_BACKUP}"
    
    # 1. Back up all Parquet data tables and checkpoints
    !cp -u *.parquet "{DRIVE_BACKUP}/" 2>/dev/null || true
    
    # 2. Back up Preprocessed matrices
    !cp -r -u Preprocess/ "{DRIVE_BACKUP}/" 2>/dev/null || true
    
    # 3. Back up GeoTIFF spatial maps
    !cp -r -u Tiff/ "{DRIVE_BACKUP}/" 2>/dev/null || true
    
    # 4. Back up MLflow tracking runs
    !cp -r -u mlruns/ "{DRIVE_BACKUP}/" 2>/dev/null || true
    
    # 5. Back up CSV and Markdown metric reports
    !cp -u *.csv *.md "{DRIVE_BACKUP}/" 2>/dev/null || true
    
    print(f"✅ All datasets, checkpoints, GeoTIFFs, MLflow runs, and metrics backed up to: {DRIVE_BACKUP}")
else:
    print("Local execution: All files stored in local workspace.")
